In [ ]:
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
preferred_split = "test"
fallback_split = "train[:2000]"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "preferred_split": preferred_split,
    "fallback_split": fallback_split,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
candidate_splits = [preferred_split, fallback_split]
selected_split = None
df = None
labels_available = False

for split_name in candidate_splits:
    ds = load_dataset(dataset_name, dataset_config, split=split_name)
    candidate_df = ds.to_pandas().copy()
    required_columns = ["sentence1", "sentence2"]
    has_required_columns = all(col in candidate_df.columns for col in required_columns)
    has_labels = "label" in candidate_df.columns and candidate_df["label"].notna().all()
    if has_required_columns and has_labels:
        selected_split = split_name
        df = candidate_df[["sentence1", "sentence2", "label"]].copy()
        labels_available = True
        break

if df is None:
    raise ValueError("No suitable split with sentence1, sentence2, and non-null label was found.")

print({
    "selected_split": selected_split,
    "labels_available": labels_available,
    "num_examples": len(df),
    "columns": df.columns.tolist(),
})
print(df.head())

In [ ]:
sentences1 = df["sentence1"].astype(str).tolist()
sentences2 = df["sentence2"].astype(str).tolist()
labels = df["label"].to_numpy(dtype=np.float32)

all_sentences = sentences1 + sentences2
sentence_counts = Counter(all_sentences)
unique_sentences = list(dict.fromkeys(all_sentences))
sentence_to_idx = {text: idx for idx, text in enumerate(unique_sentences)}

pair_idx1 = np.fromiter((sentence_to_idx[s] for s in sentences1), dtype=np.int32, count=len(sentences1))
pair_idx2 = np.fromiter((sentence_to_idx[s] for s in sentences2), dtype=np.int32, count=len(sentences2))

total_sentence_occurrences = len(all_sentences)
num_unique_sentences = len(unique_sentences)
cache_hits = total_sentence_occurrences - num_unique_sentences
cache_hit_rate = cache_hits / total_sentence_occurrences if total_sentence_occurrences else 0.0

print({
    "total_sentence_occurrences": total_sentence_occurrences,
    "num_unique_sentences": num_unique_sentences,
    "cache_hits": cache_hits,
    "cache_hit_rate": round(cache_hit_rate, 6),
    "num_reused_sentences": int(sum(count > 1 for count in sentence_counts.values())),
})

In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()

unique_embeddings = model.encode(
    unique_sentences,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

print({
    "model_name": model_name,
    "embedding_shape": tuple(unique_embeddings.shape),
})

In [ ]:
emb1 = unique_embeddings[pair_idx1]
emb2 = unique_embeddings[pair_idx2]

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)

results_df = df.copy()
results_df["sentence1_idx"] = pair_idx1
results_df["sentence2_idx"] = pair_idx2
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5

cosine_distribution = {
    "count": int(len(cosine_similarity)),
    "mean": float(np.mean(cosine_similarity)),
    "std": float(np.std(cosine_similarity)),
    "min": float(np.min(cosine_similarity)),
    "p01": float(np.quantile(cosine_similarity, 0.01)),
    "p05": float(np.quantile(cosine_similarity, 0.05)),
    "p25": float(np.quantile(cosine_similarity, 0.25)),
    "median": float(np.median(cosine_similarity)),
    "p75": float(np.quantile(cosine_similarity, 0.75)),
    "p95": float(np.quantile(cosine_similarity, 0.95)),
    "p99": float(np.quantile(cosine_similarity, 0.99)),
    "max": float(np.max(cosine_similarity)),
}

top_scored_pairs = results_df.sort_values(
    by=["cosine_similarity", "label"], ascending=[False, False]
).reset_index(drop=True).head(10)

bottom_scored_pairs = results_df.sort_values(
    by=["cosine_similarity", "label"], ascending=[True, True]
).reset_index(drop=True).head(10)

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5"]].head(10))
print(cosine_distribution)
print(top_scored_pairs[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5"]])
print(bottom_scored_pairs[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5"]])

In [ ]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{selected_split}")
print(f"num_examples: {len(df)}")
print(f"total_sentence_occurrences: {total_sentence_occurrences}")
print(f"num_unique_sentences: {num_unique_sentences}")
print(f"cache_hits: {cache_hits}")
print(f"cache_hit_rate: {cache_hit_rate:.6f}")
print(f"cosine_mean: {cosine_distribution['mean']:.6f}")
print(f"cosine_std: {cosine_distribution['std']:.6f}")
print(f"cosine_min: {cosine_distribution['min']:.6f}")
print(f"cosine_median: {cosine_distribution['median']:.6f}")
print(f"cosine_max: {cosine_distribution['max']:.6f}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print("top_scored_pairs:")
print(top_scored_pairs[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5"]].to_dict(orient="records"))
print("bottom_scored_pairs:")
print(bottom_scored_pairs[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5"]].to_dict(orient="records"))
print(f"runtime_seconds: {runtime_seconds:.2f}")